# Yeah, I'm using typescript for data cleaning in a Jupyter Notebook and python for a backend service. Byte Me!

# Split the multi-sheet xlsx file into CSV files

In [ ]:
import * as XLSXModule from "npm:xlsx@0.18.5";
import { join } from "jsr:@std/path@1";

const XLSX = (XLSXModule as any).default ?? XLSXModule;

const xlsxPath = "./raw_data/Sunday School Rank and File_ 2025-26.xlsx";
const splitDir = "./split_data";

await Deno.mkdir(splitDir, { recursive: true });

function safeFilename(name: string): string {
  return name.replace(/[\\/*?:"<>|]/g, "_").trim() || "sheet";
}

const fileBytes = await Deno.readFile(xlsxPath);

const workbook = XLSX.read(fileBytes, {
  type: "array",
});

for (const sheetName of workbook.SheetNames) {
  const worksheet = workbook.Sheets[sheetName];

  const csv = XLSX.utils.sheet_to_csv(worksheet, {
    FS: ",",
    RS: "\n",
    blankrows: false,
  });

  const csvPath = join(splitDir, `${safeFilename(sheetName)}.csv`);

  await Deno.writeTextFile(csvPath, csv);
}

# Parse regular season picks into the correct format

In [ ]:
import { getTeamID } from '../types/teams.ts';
import type { TeamID } from '../types/teams.ts';
import type { GamePick, RegularSeasonPicksRecord } from '../types/submissions.ts';
import { RegularSeasonPicksRecordSchema } from '../types/submissions.ts';

// ── Wrapper that throws on unknown team names ─────────────────────────────────

function toTeamId(name: string, ctx: string): TeamID {
  const id = getTeamID(name);
  if (!id) throw new Error(`Unknown team name: "${name}" (${ctx})`);
  return id;
}

// ── Interface CSV parser ──────────────────────────────────────────────────────

interface WeekMatchups {
  rankMatchups: [TeamID, TeamID][];
  fileMatchups: [TeamID, TeamID][];
}

function parseInterfaceCsv(text: string, filePath: string): WeekMatchups {
  const rankMatchups: [TeamID, TeamID][] = [];
  const fileMatchups: [TeamID, TeamID][] = [];
  const rows = text.split('\n').map((r) => r.split(',').map((c) => c.trim()));
  // Skip header row; each data row: Type, Away, Home, Winner
  for (const row of rows.slice(1)) {
    const type = row[0]?.toLowerCase();
    // type.endsWith('file') catches 'file', 'thanksgiving file', 'christmas file', etc.
    if (type === 'rank' || type.endsWith('file')) {
      const away = toTeamId(row[1], `${filePath} away`);
      const home = toTeamId(row[2], `${filePath} home`);
      if (type === 'rank') rankMatchups.push([away, home]);
      else fileMatchups.push([away, home]);
    }
  }
  return { rankMatchups, fileMatchups };
}

// ── Per-file parser ───────────────────────────────────────────────────────────

function parseWeekCsv(
  text: string,
  week: string,
  filePath: string,
  matchups: WeekMatchups,
): RegularSeasonPicksRecord[] {
  const ctx = filePath;
  const rows = text.split('\n').map((r) => r.split(',').map((c) => c.trim()));

  const sep = rows.findIndex((r) => r[0] === 'Picks by Rank Level');
  if (sep === -1) throw new Error(`${ctx}: missing "Picks by Rank Level" section`);

  const s1 = rows.slice(0, sep);
  const s2 = rows.slice(sep);

  const s1Header = s1[1];
  // Skip rows where player name is "#N/A" (player did not submit that week)
  const s1Players = s1.slice(2).filter((r) => r[0] !== '' && r[1] !== '#N/A');

  const s2Header = s2[1];
  const s2Players = s2.slice(2).filter((r) => r[0] !== '' && r[1] !== '#N/A');

  // Build s1 lookup: username → { rankGames, fileGames }
  interface S1Entry {
    rankGames: TeamID[];
    fileGames: TeamID[];
  }
  const s1Map = new Map<string, S1Entry>();

  for (const row of s1Players) {
    const username = row[0];
    const rankGames: TeamID[] = [];
    const fileGames: TeamID[] = [];
    for (let col = 2; col < s1Header.length; col++) {
      const label = s1Header[col];
      const cell = row[col] ?? '';
      if (cell === '') continue;
      if (label.startsWith('Rank Game')) rankGames.push(toTeamId(cell, `${ctx} s1 ${username} col${col}`));
      else if (label.startsWith('File Game')) fileGames.push(toTeamId(cell, `${ctx} s1 ${username} col${col}`));
    }
    s1Map.set(username, { rankGames, fileGames });
  }

  // Build s2 lookup: username → teams at rank levels [0]=1pt … [N-1]=N pts
  const s2Map = new Map<string, TeamID[]>();

  for (const row of s2Players) {
    const username = row[0];
    const levels: TeamID[] = [];
    for (let col = 2; col < s2Header.length; col++) {
      const label = s2Header[col];
      if (!label.startsWith('Rank Level')) continue;
      const cell = row[col] ?? '';
      if (cell === '') break;
      levels.push(toTeamId(cell, `${ctx} s2 ${username} col${col}`));
    }
    s2Map.set(username, levels);
  }

  const { rankMatchups, fileMatchups } = matchups;

  const YEAR = '2025';
  const SEASON_TYPE = '2'; // ESPN regular season type
  const submitted_at = new Date().toISOString();

  const records: RegularSeasonPicksRecord[] = [];

  for (const [username, s1Entry] of s1Map) {
    const levels = s2Map.get(username);
    if (!levels) throw new Error(`${ctx}: no rank-level data for "${username}"`);

    // levels may be empty: player missed the rank deadline but still filed a pick.
    // The schema allows rankedPicks: [] for this case.
    if (levels.length === 0) {
      console.error(`  Note: ${username} submitted file-only picks (no ranked picks)`);
    }

    // Reverse levels so [0] = most confident (highest rank level = most pts)
    // Last year's data has no real ESPN game IDs; synthesize a placeholder
    // gameId of `${away}-${home}` so it can be backfilled later.
    const rankedPicks: GamePick[] = [...levels].reverse().map((teamId) => {
      const slot = s1Entry.rankGames.indexOf(teamId);
      if (slot === -1) throw new Error(`${ctx}: ${username}: rank-level team "${teamId}" not found in game picks`);
      const [away, home] = rankMatchups[slot];
      return { gameId: `${away}-${home}`, winner: teamId };
    });

    const filedPicks: GamePick[] = s1Entry.fileGames.map((teamId, i) => {
      const [away, home] = fileMatchups[i];
      return { gameId: `${away}-${home}`, winner: teamId };
    });

    const pk = `USER#${username}`;
    const sk = `SEASON#${YEAR}#TYPE#${SEASON_TYPE}#WEEK#${week}`;

    records.push(
      RegularSeasonPicksRecordSchema.parse({
        pk,
        sk,
        gsi1pk: sk,
        gsi1sk: pk,
        userId: username,
        year: YEAR,
        season_type: SEASON_TYPE,
        week,
        kind: 'regular',
        picks: { rankedPicks, filedPicks },
        submitted_at,
      }),
    );
  }

  return records;
}

// ── Main: scan directory, process all matching files ─────────────────────────

const splitDataDir = './split_data/';
const weekPattern = /^Week (\d+) Picks\.csv$/;

const allRecords: RegularSeasonPicksRecord[] = [];

const entries: { week: number; name: string }[] = [];

for await (const entry of Deno.readDir(splitDataDir)) {
  if (!entry.isFile) continue;
  const match = entry.name.match(weekPattern);
  if (!match) continue;
  entries.push({ week: parseInt(match[1], 10), name: entry.name });
}

// Process in week order
entries.sort((a, b) => a.week - b.week);

for (const { week, name } of entries) {
  const picksPath = `./split_data/${name}`;
  const interfaceName = `Week ${week} Interface.csv`;
  const interfacePath = `./split_data/${interfaceName}`;

  const picksText = await Deno.readTextFile(picksPath);
  const interfaceText = await Deno.readTextFile(interfacePath);

  const matchups = parseInterfaceCsv(interfaceText, interfaceName);
  const records = parseWeekCsv(picksText, String(week), name, matchups);
  allRecords.push(...records);
  console.error(`  Week ${week}: ${records.length} player records`);
}

console.error(`\nTotal records: ${allRecords.length}`);
await Deno.mkdir('clean_data', { recursive: true });
const regularSeasonPicksPath = 'clean_data/regularSeasonPicks.json';
await Deno.writeTextFile(regularSeasonPicksPath, JSON.stringify(allRecords, null, 2));


# Parse playoff picks into the correct format

In [ ]:
import type { PlayoffPicksRecord, PlayoffStraightBet, PlayoffParlayLeg, PlayoffParlayBet } from '../types/submissions.ts';
import { PlayoffPicksRecordSchema } from '../types/submissions.ts';

// ── Known typos in the source spreadsheet ────────────────────────────────────

const PICK_TYPOS: Record<string, string> = {
  'panters': 'panthers', // KevinsTeam, Wildcard Round
};

// ── Extract a TeamID from a raw string ───────────────────────────────────────
// Handles: "Panthers (+10.5)", "49ers @ Eagles (-4.5) Sun.", "Broncos: Sun."

function extractTeam(raw: string): TeamID {
  // Strip spread notation and everything after a colon (time info)
  let s = raw.replace(/\([^)]*\)/g, '').split(':')[0].trim();
  // Try longest-first word combinations so "San Francisco" beats "San"
  const words = s.split(/\s+/).filter(Boolean);
  for (let n = words.length; n >= 1; n--) {
    const candidate = words.slice(0, n).join(' ');
    const corrected = PICK_TYPOS[candidate.toLowerCase()] ?? candidate;
    const id = getTeamID(corrected);
    if (id) return id;
  }
  throw new Error(`Cannot extract TeamID from: "${raw}"`);
}

// ── Game slot parsed from the header row ─────────────────────────────────────

interface GameSlot {
  gameId: string; // e.g. "BUF-DEN"
  away: TeamID;
  home: TeamID;
}

function parseGameSlots(headerRow: string[], gameStartCol: number, colSpan: number): GameSlot[] {
  const slots: GameSlot[] = [];
  for (let col = gameStartCol; col < headerRow.length; col += colSpan) {
    const desc = headerRow[col]?.trim();
    if (!desc) break;
    const atIdx = desc.indexOf('@');
    if (atIdx === -1) break;
    const away = extractTeam(desc.slice(0, atIdx));
    const home = extractTeam(desc.slice(atIdx + 1));
    slots.push({ gameId: `${away}-${home}`, away, home });
  }
  return slots;
}

// ── Validate picked team belongs to a game slot ───────────────────────────────

function assertPickedTeam(picked: TeamID, slot: GameSlot, ctx: string): TeamID {
  if (picked === slot.home || picked === slot.away) return picked;
  throw new Error(`${ctx}: "${picked}" not in game ${slot.gameId} (${slot.away} @ ${slot.home})`);
}

// ── Parse one playoff round CSV ───────────────────────────────────────────────
//
// WC/Divisional format: game headers at col 8+, 4-col groups (SinglePick, Bet, Outcome, ParlayPick).
// Championship format:  game headers at col 5+, 3-col groups (SinglePick, Bet, Outcome). No parlay.
// Detected by whether col 3 of the header row contains "Parlay Bet".

function parsePlayoffRoundCsv(text: string, week: string, filePath: string): PlayoffPicksRecord[] {
  const rows = text.split('\n').map((r) => r.split(',').map((c) => c.trim()));
  const headerRow = rows[0];
  const colHeaders = rows[1];

  const hasParlay = colHeaders[3]?.toLowerCase().includes('parlay');
  const gameStartCol = hasParlay ? 8 : 5;
  const colSpan = hasParlay ? 4 : 3;

  const slots = parseGameSlots(headerRow, gameStartCol, colSpan);
  if (slots.length === 0) throw new Error(`${filePath}: no game slots found in header row`);

  const YEAR = '2025';
  const SEASON_TYPE = '3';
  const submitted_at = new Date().toISOString();
  const records: PlayoffPicksRecord[] = [];

  for (const row of rows.slice(2)) {
    const username = row[0]?.trim();
    if (!username) continue;

    const ctx = `${filePath} / ${username}`;
    const straightBets: PlayoffStraightBet[] = [];
    const parlayLegs: PlayoffParlayLeg[] = [];

    for (let i = 0; i < slots.length; i++) {
      const slot = slots[i];
      const base = gameStartCol + i * colSpan;

      const singlePickStr = row[base]?.trim() ?? '';
      const singleBetStr = (row[base + 1]?.trim() ?? '').replace(/,/g, '');
      if (singlePickStr && singleBetStr) {
        const amount = parseInt(singleBetStr, 10);
        if (amount > 0) {
          const picked = extractTeam(singlePickStr);
          straightBets.push({ gameId: slot.gameId, winner: assertPickedTeam(picked, slot, ctx), amount });
        }
      }

      if (hasParlay) {
        const parlayPickStr = row[base + 3]?.trim() ?? '';
        if (parlayPickStr) {
          const picked = extractTeam(parlayPickStr);
          parlayLegs.push({ gameId: slot.gameId, winner: assertPickedTeam(picked, slot, ctx) });
        }
      }
    }

    // Build parlay bet only if >= 2 legs and a positive stake
    let parlayBet: PlayoffParlayBet | null = null;
    if (hasParlay && parlayLegs.length >= 2) {
      const amount = parseInt((row[3]?.trim() ?? '').replace(/,/g, ''), 10);
      if (amount > 0) parlayBet = { legs: parlayLegs, amount };
    }

    if (straightBets.length === 0 && parlayBet === null) {
      console.error(`  Note: ${username} placed no bets this round`);
      continue;
    }

    const pk = `USER#${username}`;
    const sk = `SEASON#${YEAR}#TYPE#${SEASON_TYPE}#WEEK#${week}`;

    records.push(
      PlayoffPicksRecordSchema.parse({
        pk,
        sk,
        gsi1pk: sk,
        gsi1sk: pk,
        userId: username,
        year: YEAR,
        season_type: SEASON_TYPE,
        week,
        kind: 'playoff',
        picks: { straightBets, parlayBet },
        submitted_at,
      }),
    );
  }

  return records;
}

// ── Round configuration ───────────────────────────────────────────────────────

const playoffRounds = [
  { file: 'Wildcard Round.csv',     week: '1', out: 'clean_data/playoffPicks_wildcard.json' },
  { file: 'Divisional Round.csv',   week: '2', out: 'clean_data/playoffPicks_divisional.json' },
  { file: 'Championship Round.csv', week: '3', out: 'clean_data/playoffPicks_championship.json' },
];

await Deno.mkdir('clean_data', { recursive: true });

for (const { file, week, out } of playoffRounds) {
  const text = await Deno.readTextFile(`./split_data/${file}`);
  const records = parsePlayoffRoundCsv(text, week, file);
  console.error(`  ${file}: ${records.length} player records`);
  await Deno.writeTextFile(out, JSON.stringify(records, null, 2));
}